# FIT5226 Project Stage 2: Multi-Agent Stochastic Games
**Student Name:** Juan Pulido  
**Student ID:** 34169202  
**Student Email:** jpul0007@student.monash.edu  


# 📖 Table of Contents

## Phase I: Foundation
1. AI Declaration
2. Solution Architecture
3. Mathematical Formulation (MMDP & EGT)
## Phase II: Implementation (Standalone)
4. Dependency Installation
5. Imports & Configuration Layer
6. Core State & Action Handlers
7. Environment Layer (Simultaneous Steps & Collision Physics)
8. Agent Layer (Sarah's Expected Value Bellman Update)
9. Orchestration Layer (Simulation Runner)
## Phase III: Training & Experimentation
10. Model Training & Execution
11. HD Experiment: Phase 2 Step Cost Tipping Point
## Phase IV: Mathematical Modeling
12. Evolutionary Game Theory (EGT) Simulation
## Phase V: Conclusion
13. Theoretical Justification (Task 4)
---


# Phase I: Foundation

## 1. AI Declaration

In accordance with the unit's Generative AI policy, I declare that Generative AI (specifically Gemini) was used in the preparation of this notebook as an Architectural Co-Pilot, Technical Auditor, and coding assistant for non-core utility functions. 

Specifically, AI was used to assist with the following components:
*   **Code Refactoring:** Assisted in stripping out Stage 1 legacy code and refactoring the environment for simultaneous multi-agent execution.
*   **EGT Simulation:** Assisted in writing the `scipy.integrate.odeint` syntax for the Replicator Dynamics numerical simulation and matplotlib generation.
*   **Literate Programming:** Assisted in structuring the Markdown cells and formatting LaTeX mathematical equations.

**Verification Statement:**
I have manually reviewed and evaluated all code provided by AI. I confirm that the core algorithmic logic, specifically the physics collision engine and **Sarah's Expected Value Bellman Update**, was manually verified against the unit lectures. I take full responsibility for the accuracy and integrity of this submission.


## 2. Solution Architecture
This solution builds upon the modular architecture of Stage 1 but introduces critical complexity to handle Multi-Agent Reinforcement Learning (MARL).

**NOTE:** To satisfy the submission constraints, the entire modular project architecture (Configuration, Domain, Core, Agents, Orchestration) has been flattened into this single standalone Jupyter Notebook. No external `.py` or `.json` files are required to run this solution.

The implementation is partitioned into these core logical layers:
1. **Configuration Layer:** Centralises hyperparameter dictionaries, strictly defining Phase 1 vs. Phase 2 (symmetric vs. asymmetric) reward penalties inline.
2. **Environment Layer (`StochasticMultiAgentEnv`):** Manages the true global state. It resolves intended moves *simultaneously* before committing state changes, and processes the stochastic toggling of the lake at $(2,2)$.
3. **Agent Layer (`TabularQAgent`):** Implements Sarah's refined Tabular Q-Learning agent, utilizing a custom Expected Value Bellman update that leverages the known probability $P$ of the lake's state transition.
4. **Orchestration Layer (`SimulationRunner`):** Manages the decentralized action selection and learning loops for both Agent A and Agent B.

## 3. Mathematical Formulation

### Multi-Agent Markov Decision Process (MMDP)
We formalise this transport task as an MMDP.
* **State Space ($S$):** The global state contains positions of A and B, their payload status, and the binary lake state. However, agents are *partially observable*; they only see their own position, payload, and the lake state. 
* **Action Space ($A$):** Reduced to 5 discrete actions (NORTH, SOUTH, EAST, WEST, WAIT).
* **Transition Dynamics ($P$):** The movement is deterministic, but the environment features **Exogenous Stochastic State Transitions**. The lake at $(2,2)$ toggles flooded/dry with a known probability $p$.

### Game Theoretic Abstract (Intersection Conflict)
In Phase 2 (Symmetric rewards), the interaction at the intersection $(2,2)$ collapses into a 2-player Normal-Form Anti-Coordination Game. If both agents attempt to 'Cross' (C), they suffer a massive collision penalty. If both 'Wait' (W), they suffer delay costs. Standard independent Q-learners face severe non-stationarity trying to adapt to each other in this symmetric trap.

To validate the population convergence and stability of this interaction, we utilize **Evolutionary Game Theory** and model the population growth using the **Replicator Dynamics Equation** (as formalized in Week 8 Lecture, *Population Games & Evolution Game Theory*):
$$\dot{x}_i = x_i \left( (Ax)_i - x^T Ax \right)$$
Where $\dot{x}_i$ is the per capita growth rate of strategy $i$, $(Ax)_i$ is the fitness (average payoff) of the strategy, and $x^T Ax$ is the mean fitness across the population. This allows us to empirically simulate if the 'Cross' strategy achieves an Evolutionarily Stable Strategy (ESS).

# Phase II: Implementation (Standalone)

## 4. Dependency Installation
To ensure this standalone notebook runs flawlessly in any environment, we first install the required mathematical and visualization libraries.

In [ ]:
# Install required libraries quietly
%pip install -q numpy matplotlib scipy

## 5. Imports & Configuration Layer
We pull in our python dependencies and build the configuration classes. Notice the `action_size` is now 5, and we define our Phase 2 parameters inline as a python dictionary rather than reading from a JSON file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import defaultdict
from dataclasses import dataclass, field, replace
from enum import IntEnum
from typing import Tuple, List, Dict, Any, Optional
from abc import ABC, abstractmethod
from scipy.integrate import odeint

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)

### CONFIGURATION CLASSES ###

@dataclass(frozen=True)
class AgentConfig:
    """Hyperparameters for the Tabular Q-Learning Agent."""
    learning_rate_alpha: float
    discount_factor_gamma: float
    initial_epsilon: float
    epsilon_decay_rate: float
    minimum_epsilon: float
    action_size: int  # 5 for Stage 2 (N, S, E, W, WAIT)

@dataclass(frozen=True)
class EnvConfig:
    """Structural parameters and reward values for the Rugged Planet MARL GridWorld."""
    grid_rows: int
    grid_cols: int
    p_flood: float  # Probability of lake flooding (Stochasticity)
    step_cost: float
    success_reward: float
    collision_penalty: float = 0.0
    hazard_penalty: float = 0.0

@dataclass(frozen=True)
class ExperimentConfig:
    """The root configuration object for a specific simulation run."""
    experiment_name: str
    is_multi_agent: bool
    training_episode_budget: int
    agent: AgentConfig
    env: EnvConfig

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> 'ExperimentConfig':
        """
        Loads an experiment configuration from a dictionary.
        """
        return cls(
            experiment_name=data.get("experiment_name", "Unnamed_Experiment"),
            is_multi_agent=data.get("is_multi_agent", True),
            training_episode_budget=data["training_episode_budget"],
            agent=AgentConfig(**data["agent"]),
            env=EnvConfig(**data["env"])
        )

### INLINE JSON CONFIGURATION ###

STAGE2_PHASE2_CONFIG = {
  "experiment_name": "Stage2_Phase2_Sarah_Safe",
  "is_multi_agent": True,
  "training_episode_budget": 5000,
  "agent": {
    "learning_rate_alpha": 0.1,
    "discount_factor_gamma": 0.99,
    "initial_epsilon": 1.0,
    "epsilon_decay_rate": 0.9995,
    "minimum_epsilon": 0.01,
    "action_size": 5
  },
  "env": {
    "grid_rows": 5,
    "grid_cols": 5,
    "p_flood": 0.5,
    "step_cost": -5.0,
    "success_reward": 50.0,
    "collision_penalty": -20.0,
    "hazard_penalty": 0.0
  }
}

config = ExperimentConfig.from_dict(STAGE2_PHASE2_CONFIG)
print(f"Loaded Config: {config.experiment_name}")
print(f"Grid: {config.env.grid_rows}x{config.env.grid_cols}, p_flood: {config.env.p_flood}")
print(f"Penalties -> Step: {config.env.step_cost}, Collision: {config.env.collision_penalty}, Hazard: {config.env.hazard_penalty}")

## 5. Core State & Action Handlers
Defining the physical movement deltas and how observations are parsed.

In [ ]:
class Directions(IntEnum):
    """
    Semantic enumeration of cardinal directions and WAIT.
    Specifically for Stage 2 Multi-Agent Stochastic Games.
    """
    NORTH = 0
    SOUTH = 1
    EAST = 2
    WEST = 3
    WAIT = 4

class Actions:
    """
    Handles coordinate geometry using (y, x) indexing for the Stage 2 grid.
    - (0, 2) is North (Y)
    - (4, 2) is South (V)
    - (2, 0) is West (X)
    - (2, 4) is East (U)
    - (2, 2) is the Lake
    """
    _DELTAS: Dict[Directions, Tuple[int, int]] = {
        Directions.NORTH: (-1, 0),
        Directions.SOUTH: (1, 0),
        Directions.EAST: (0, 1),
        Directions.WEST: (0, -1),
        Directions.WAIT: (0, 0),
    }

    @staticmethod
    def get_delta(action: int) -> Tuple[int, int]:
        direction = Directions(action)
        return Actions._DELTAS.get(direction, (0, 0))

    @staticmethod
    def apply_action(current_pos: Tuple[int, int], action: int) -> Tuple[int, int]:
        dy, dx = Actions.get_delta(action)
        return (current_pos[0] + dy, current_pos[1] + dx)

    @staticmethod
    def is_valid_move(pos: Tuple[int, int], grid_size: Tuple[int, int]) -> bool:
        y, x = pos
        rows, cols = grid_size
        return 0 <= y < rows and 0 <= x < cols

class StateHandler:
    """
    Transforms raw environment observations into compact, hashable 
    state representations for Tabular Q-Learning.
    """
    @staticmethod
    def get_agent_state(agent_id: str, observation: Dict[str, Any]) -> Tuple:
        """
        Extracts the partial observation for a specific agent.
        According to Stage 2 Specs:
        - Own location (x, y)
        - Whether it carries a sample (bool)
        - Binary state of the lake (bool)
        - (Does NOT see other agents' locations)
        """
        pos = observation["positions"][agent_id]
        has_sample = observation["has_sample"][agent_id]
        lake_flooded = observation["lake_flooded"]
        
        return (pos[0], pos[1], has_sample, lake_flooded)

## 6. Environment Layer (Simultaneous Steps & Collision Physics)

A common trap in MARL gridworlds is updating Agent A, which changes the state, and then evaluating Agent B based on that intermediate state. 
Our `StochasticMultiAgentEnv` correctly calculates *intended* states for all agents first, applies boundary checks, and then resolves collisions if `positions["Agent_A"] == positions["Agent_B"]`. It also stochastically flips the lake state *before* movement resolution.

In [ ]:
class BaseEnvironment(ABC):
    @abstractmethod
    def reset(self) -> Dict[str, Any]:
        pass

    @abstractmethod
    def step(self, joint_action: Dict[str, int]) -> Tuple[Dict[str, Any], Dict[str, float], Dict[str, bool], bool]:
        pass

class StochasticMultiAgentEnv(BaseEnvironment):
    """
    Stage 2 GridWorld: Multi-agent, simultaneous steps, stochastic lake.
    Refactored for Industry/Academic standards using Actions class.
    """
    def __init__(self, config: EnvConfig):
        self.config = config
        self.grid_size = (config.grid_rows, config.grid_cols)
        
        # Locations (y, x)
        self.locs = {
            "X": (2, 0),
            "Y": (0, 2),
            "U": (2, 4),
            "V": (4, 2),
            "Lake": (2, 2)
        }
        
        self.agent_types = {
            "Agent_A": "Type_A", # X -> U -> X
            "Agent_B": "Type_B"  # Y -> V -> Y
        }
        
        self.reset()

    def reset(self) -> Dict[str, Any]:
        self.positions = {
            "Agent_A": self.locs["X"],
            "Agent_B": self.locs["Y"]
        }
        self.has_sample = {
            "Agent_A": False,
            "Agent_B": False
        }
        self.lake_flooded = False # Initial state
        self.done = {
            "Agent_A": False,
            "Agent_B": False
        }
        return self._get_obs()

    def _get_obs(self) -> Dict[str, Any]:
        return {
            "positions": self.positions.copy(),
            "has_sample": self.has_sample.copy(),
            "lake_flooded": self.lake_flooded
        }

    def step(self, joint_action: Dict[str, int]) -> Tuple[Dict[str, Any], Dict[str, float], Dict[str, bool], bool]:
        rewards = {"Agent_A": 0.0, "Agent_B": 0.0}
        
        # 1. Stochastic Lake Transition (happens before movement resolution)
        if random.random() < self.config.p_flood:
            self.lake_flooded = not self.lake_flooded

        # 2. Movement Resolution (Simultaneous)
        prev_positions = self.positions.copy()
        new_positions = {}
        
        for agent_id, action in joint_action.items():
            if self.done[agent_id]:
                new_positions[agent_id] = prev_positions[agent_id]
                continue
                
            # Use Actions class for coordinate math
            potential_pos = Actions.apply_action(prev_positions[agent_id], action)
            
            if Actions.is_valid_move(potential_pos, self.grid_size):
                new_positions[agent_id] = potential_pos
            else:
                new_positions[agent_id] = prev_positions[agent_id]
            
            # Step/Wait Costs
            if action == Directions.WAIT:
                rewards[agent_id] += self.config.step_cost * 0.6 # -3 vs -5
            else:
                rewards[agent_id] += self.config.step_cost # -5

        self.positions = new_positions

        # 3. Collision & Water Hazard Detection
        # Collision: agents occupy the same cell anywhere on the grid
        if self.positions["Agent_A"] == self.positions["Agent_B"]:
            rewards["Agent_A"] += self.config.collision_penalty
            rewards["Agent_B"] += self.config.collision_penalty

        # Water Hazard (Agent A only, only in Phase 1)
        if self.lake_flooded and self.positions["Agent_A"] == self.locs["Lake"]:
            rewards["Agent_A"] += self.config.hazard_penalty

        # 4. Task Progress (Pickup / Delivery)
        for agent_id in ["Agent_A", "Agent_B"]:
            if self.done[agent_id]: continue
            
            pos = self.positions[agent_id]
            atype = self.agent_types[agent_id]
            
            if atype == "Type_A":
                # Pickup at U
                if not self.has_sample[agent_id] and pos == self.locs["U"]:
                    self.has_sample[agent_id] = True
                    rewards[agent_id] += 10.0
                # Deliver at X
                elif self.has_sample[agent_id] and pos == self.locs["X"]:
                    self.done[agent_id] = True
                    rewards[agent_id] += self.config.success_reward
            else: # Type B
                # Pickup at V
                if not self.has_sample[agent_id] and pos == self.locs["V"]:
                    self.has_sample[agent_id] = True
                    rewards[agent_id] += 10.0
                # Deliver at Y
                elif self.has_sample[agent_id] and pos == self.locs["Y"]:
                    self.done[agent_id] = True
                    rewards[agent_id] += self.config.success_reward

        all_done = all(self.done.values())
        return self._get_obs(), rewards, self.done, all_done

## 7. Agent Layer (Sarah's Expected Value Bellman Update)

Standard off-policy **Q-Learning** (as formalized in Week 4, *SARSA and Q-Learning*) uses a single-sample Temporal Difference (TD) target backed by the greedy action in the observed next state:
$$Q(S, A) \leftarrow Q(S, A) + \alpha \left( R + \gamma \max_{a'} Q(S', a') - Q(S, A) \right)$$

However, because Sarah knows the environmental transition probability $p$ of the lake changing state, her agent shouldn't blindly trust the single observed next state $S'$. Instead, she computes the *expected* maximum future value over the two possible universe branches to stabilize learning in the stochastic environment:

$$ \text{Expected Future Q} = (1 - p) \max_{a'} Q(S_{\text{unchanged}}, a') + (p) \max_{a'} Q(S_{\text{flipped}}, a') $$

This hybrid model-aware update is implemented in `TabularQAgent.update_learning()`.

In [ ]:
class BaseAgent(ABC):
    def __init__(self, agent_id: str, config: Any):
        self.agent_id = agent_id
        self.config = config
        
    @abstractmethod
    def choose_action(self, observation: Any) -> int:
        pass

    @abstractmethod
    def update_learning(self, state: Any, action: int, reward: float, next_state: Any, terminal: bool) -> None:
        pass

class TabularQAgent(BaseAgent):
    """
    Agent implementation using the Tabular Q-Learning algorithm.
    Refactored to match Stage 2 Multi-Agent requirements.
    """
    def __init__(self, agent_id: str, config: AgentConfig, p_flood: float = 0.0):
        super().__init__(agent_id, config)
        self.p_flood = p_flood
        
        # Q-Table Initialization: Q(s, a)
        # Using a sparse mapping (defaultdict)
        self.q_table = defaultdict(lambda: np.zeros(self.config.action_size))

        # Exploration rate (epsilon)
        self.epsilon = self.config.initial_epsilon

    def choose_action(self, observation: Any) -> int:
        """
        Epsilon-greedy action selection.
        Observation should be the hashable state tuple from StateHandler.
        """
        if np.random.rand() < self.epsilon:
            return int(np.random.randint(self.config.action_size))

        return int(np.argmax(self.q_table[observation]))

    def update_learning(self,
                        state: Tuple,
                        action: int,
                        reward: float,
                        next_obs_state: Tuple,
                        terminal: bool) -> None:
        """
        Sarah's Refined Bellman Update: Accounts for the known transition 
        probability 'p' of the lake's stochastic state.
        """
        p = self.p_flood # Transition probability
        current_q = self.q_table[state][action]
        
        if terminal:
            expected_future_q = 0.0
        else:
            # State = (y, x, has_payload, lake_flooded)
            y, x, payload, lake = next_obs_state
            state_unchanged = (y, x, payload, lake)
            state_flipped = (y, x, payload, not lake)
            
            max_q_unchanged = np.max(self.q_table[state_unchanged])
            max_q_flipped = np.max(self.q_table[state_flipped])
            
            # Expected Value over the transition model
            expected_future_q = ( (1 - p) * max_q_unchanged ) + ( p * max_q_flipped )

        # TD Target using Expected Future Value
        td_target = reward + (self.config.discount_factor_gamma * expected_future_q)
        self.q_table[state][action] += self.config.learning_rate_alpha * (td_target - current_q)

    def decay_epsilon(self):
        """Standard exponential decay of the exploration rate."""
        self.epsilon = max(
            self.config.minimum_epsilon,
            self.epsilon * self.config.epsilon_decay_rate
        )

## 8. Orchestration Layer (Simulation Runner)
Manages the decentralized action selection and learning loops for both Agent A and Agent B.

In [ ]:
class SimulationRunner:
    """
    Orchestrates the Multi-Agent training loop.
    """
    def __init__(self, config: ExperimentConfig, env: Any, agents: Dict[str, Any]):
        self.config = config
        self.env = env
        self.agents = agents

    def run_experiment(self):
        """
        Executes the training loop over the specified episode budget.
        """
        print(f"--- Starting Experiment: {self.config.experiment_name} ---")

        for episode in range(self.config.training_episode_budget):
            obs = self.env.reset()
            
            episode_dones = {aid: False for aid in self.agents.keys()}
            total_rewards = {aid: 0.0 for aid in self.agents.keys()}
            step_count = 0
            collisions = 0

            while not all(episode_dones.values()):
                # 1. Action Selection (Decentralized)
                joint_action = {}
                for aid, agent in self.agents.items():
                    if not episode_dones[aid]:
                        state_key = StateHandler.get_agent_state(aid, obs)
                        joint_action[aid] = agent.choose_action(state_key)
                    else:
                        # If agent is done, it waits
                        joint_action[aid] = 4 

                # 2. Environment Step
                next_obs, rewards, dones, truncated = self.env.step(joint_action)

                # 3. Learning (Decentralized)
                for aid, agent in self.agents.items():
                    if not episode_dones[aid]:
                        state_key = StateHandler.get_agent_state(aid, obs)
                        next_state_key = StateHandler.get_agent_state(aid, next_obs)
                        
                        agent.update_learning(
                            state_key,
                            joint_action[aid],
                            rewards[aid],
                            next_state_key,
                            dones[aid]
                        )
                        total_rewards[aid] += rewards[aid]
                        
                        # Collision detection for metrics
                        if rewards[aid] <= self.config.env.collision_penalty and self.config.env.collision_penalty < 0:
                            collisions += 1

                obs = next_obs
                episode_dones = dones
                step_count += 1
                if step_count > 2000: # Safety break
                    break

            # 4. Epsilon Decay (Per Episode)
            for agent in self.agents.values():
                agent.decay_epsilon()

            # Logging
            if episode > 0 and episode % 1000 == 0:
                avg_reward = np.mean(list(total_rewards.values()))
                print(f"Episode {episode:5d} | Avg Reward: {avg_reward:7.2f} | Steps: {step_count:4d} | Collisions: {collisions:3d}")

        print("--- Training Complete ---")

# Phase III: Training & Experimentation

## 9. Model Training & Execution
We run the standard simulation orchestration loop. The agents select decentralized actions, step the environment simultaneously, and apply their respective Expected Value Bellman updates.

In [ ]:
# Initialize Environment
env = StochasticMultiAgentEnv(config.env)
agent_ids = ["Agent_A", "Agent_B"]

# Initialize Agents
agents = {
    aid: TabularQAgent(agent_id=aid, config=config.agent, p_flood=config.env.p_flood) 
    for aid in agent_ids
}

# Run Orchestrator (Baseline symmetric training)
runner = SimulationRunner(config, env, agents)
runner.run_experiment()

## 10. HD Experiment: Phase 2 Step Cost Tipping Point
To demonstrate why Phase 2 is uniquely difficult, we run a structured experiment varying the step cost. As the step penalty becomes more severe, agents are forced to rush the intersection rather than safely waiting, leading to a catastrophic collapse of the Nash Equilibrium.

In [ ]:
def run_hd_experiment():
    """
    HD Experiment: Systematically modify the step penalty (delay cost) in Phase 2
    to identify the tipping point where standard Q-learning collapses from collision
    into sub-optimal 'Mutual Wait' equilibrium.
    """
    print("\n=== Running HD Experiment: Varying Step Penalties in Phase 2 ===")
    step_costs = [-1.0, -3.0, -5.0, -7.0, -10.0, -15.0]
    
    for cost in step_costs:
        # Modify the step cost structurally for the experiment
        new_env_config = replace(config.env, step_cost=cost, hazard_penalty=0.0)
        exp_config = replace(config, env=new_env_config)
        
        env = StochasticMultiAgentEnv(exp_config.env)
        agent_ids = ["Agent_A", "Agent_B"]
        
        agents = {
            aid: TabularQAgent(agent_id=aid, config=exp_config.agent, p_flood=exp_config.env.p_flood) 
            for aid in agent_ids
        }
        
        collisions = 0
        total_steps = 0
        
        for episode in range(exp_config.training_episode_budget):
            obs = env.reset()
            episode_dones = {aid: False for aid in agents.keys()}
            
            while not all(episode_dones.values()):
                joint_action = {}
                for aid, agent in agents.items():
                    if not episode_dones[aid]:
                        state_key = StateHandler.get_agent_state(aid, obs)
                        joint_action[aid] = agent.choose_action(state_key)
                    else:
                        joint_action[aid] = 4
                        
                next_obs, rewards, dones, _ = env.step(joint_action)
                
                for aid, agent in agents.items():
                    if not episode_dones[aid]:
                        state_key = StateHandler.get_agent_state(aid, obs)
                        next_state_key = StateHandler.get_agent_state(aid, next_obs)
                        agent.update_learning(state_key, joint_action[aid], rewards[aid], next_state_key, dones[aid])
                        
                        if rewards[aid] <= exp_config.env.collision_penalty and exp_config.env.collision_penalty < 0:
                            collisions += 1
                            
                obs = next_obs
                episode_dones = dones
                total_steps += 1
                
                if total_steps > 500000:
                    break
                    
            for agent in agents.values():
                agent.decay_epsilon()
                
        print(f"Step Cost: {cost:>5.1f} | Total Collisions over {exp_config.training_episode_budget} episodes: {collisions:>5}")

run_hd_experiment()

# Phase IV: Mathematical Modeling

## 11. Evolutionary Game Theory (EGT) Simulation
We isolate the interaction at the intersection (2,2) as a 2-player normal-form game. Let $C$ = Cross and $W$ = Wait. 
Using the Replicator Dynamics ODE: $\dot{x} = x(f_C - \phi)$, we numerically simulate the population convergence.

In [ ]:
def replicator_dynamics(x, t):
    """
    Computes the derivative dx/dt for the replicator dynamics ODE.
    
    x: Proportion of the population playing 'Cross' (C)
    1-x: Proportion of the population playing 'Wait' (W)
    
    Payoff Matrix (Symmetric Game in Phase 2):
    U(C, C) = -25  (Step: -5, Collision: -20)
    U(C, W) = -5   (Step: -5)
    U(W, C) = -3   (Wait: -3)
    U(W, W) = -3   (Wait: -3)
    """
    # Expected fitness for each pure strategy
    f_C = x * (-25) + (1 - x) * (-5)
    f_W = x * (-3) + (1 - x) * (-3)
    
    # Average population fitness
    phi = x * f_C + (1 - x) * f_W
    
    # Replicator equation
    dxdt = x * (f_C - phi)
    return dxdt

def run_simulation():
    """
    Simulates the replicator dynamics using scipy's ODE solver.
    Produces a visualization of population convergence.
    """
    t = np.linspace(0, 10, 200)
    initial_conditions = [0.01, 0.1, 0.5, 0.9, 0.99]
    
    plt.figure(figsize=(10, 6))
    
    for x0 in initial_conditions:
        sol = odeint(replicator_dynamics, x0, t)
        plt.plot(t, sol[:, 0], label=f"Initial C% = {x0*100:.1f}%")
        
    plt.title("Evolutionary Game Theory: Replicator Dynamics for Intersection Game", fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Proportion of Population Playing 'Cross' (C)", fontsize=12)
    plt.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='ESS (All Wait / Collapse)')
    plt.ylim(-0.05, 1.05)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend()
    plt.show()

run_simulation()

# Phase V: Conclusion

## 12. Theoretical Justification (Task 4)

In Phase 1, the water hazard naturally imposes an asymmetric game, breaking the symmetry between agents A and B. This inherent asymmetry acts as an environmental correlating device, facilitating clear equilibrium selection (Agent B crosses while Agent A waits), leading to rapid convergence. 

However, in Phase 2, the removal of the water penalty creates a perfectly symmetric game. Without an external correlating device (such as a traffic light) to coordinate joint actions, agents lack a mechanism for equilibrium selection. They face an anti-coordination problem at the intersection, continuously updating their policies based on the shifting behavior of the other agent. This mutual adaptation creates severe non-stationarity in the environment from the perspective of each independent tabular Q-learner. As a result, standard independent MARL struggles to converge, often oscillating between catastrophic collisions and sub-optimal mutual waiting.